In [ ]:
# Purpose - makes fields/spectra movie (newer)
# 9, 100

findex = 160 #140 #9 #100 #9 #100 #6 #25

In [ ]:
# Imports and Folder Selection

%load_ext autoreload
%autoreload 2

from graphet import Data
from graphet.plugins import TristanV2 
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import sys
sys.path.append('../v2_proc_files')
from useful_funcs import *
from TristanRun import TristanRun
from FieldLineHelper import FieldLineHelper 

plt.close('all')

%matplotlib widget

set_standard_plot_params()

# Read Input File Data

run = TristanRun(findex)
gyrofreq = run.gyrofreq
boilerplate = run.boilerplate + f"tstep = {run.steps[0]}"
cc = run.cc
B0 = run.B0

my_steps = {140: run.steps[70:], 160: run.steps[18:]}[findex] #[run.steps[0]]

In [ ]:
# Configure Data

d = run.config_data(my_steps, "fields")

In [ ]:

# Plot Domain and Spectrum

counter = 0
ds = 5
buffer = 10

flh = FieldLineHelper(run, 5, contour_ds = ds, origin = [0.25, 0.5]) #20, 0.1, 0.1
#flh = FieldLineHelper(run, 10, contour_ds = ds, origin = [0.25, 0.5]) #20, 0.1, 0.1

i = 0
for abs_ts in my_steps:
    print(f"abs_ts = {abs_ts}, counter = {counter}")
    plt.close()

    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10, 8))

    buffer = 5
    rect = [buffer, run.xlim - buffer, buffer, run.ylim - buffer]
    #rect = [buffer, run.xlim - buffer, 2000, run.ylim - buffer]

    roi = fq_cut(d.fields, abs_ts - my_steps[0], rect)


    mesh = plot_fq3(roi, roi.dens, plot = ax, vabsmax = 5, downsample = ds, positive = True)
    cbar1 = fig.colorbar(mesh, ax=ax)
    cbar1.set_label('$n / n_0$', rotation=270, labelpad=15)

    spare_movement = 1

    ez_rep = roi.ez.mean()
    #ez_rep = np.mean(roi.ez[0, :])
    #ez_rep = roi.ez[0, 0]
    flh.plot_field_lines(roi, ez_rep * spare_movement, arrow_spacing = 7 * 7 * 50)
    
    print(ez_rep.values.item() / (run.B0 * run.beta_A))

    xa, ya = [0.02, 0.99]

    #ax.text(xa, ya, f"$\sigma$ = {run.sigma}", color='black', ha='left', va='top',
    #        transform=ax.transAxes, fontsize = 20, weight = "bold",
    #        bbox=dict(boxstyle='square,pad=0.3', edgecolor='white', facecolor='white', alpha=0.8))
    
    plt.title(f"$\Omega t$ = {round((abs_ts - my_steps[0]) * run.gyrofreq):05d}", fontsize = 20)

    save_movie_frame(counter, dpi = 300)
    counter +=1


In [ ]:
def plot_fq4(roi, quantity, plot = None, title=None, vabsmax=-1, cmap='viridis', downsample = 1, positive = False):
    is_plt = plot is None

    if is_plt:
        plot = plt

    # Downsample the data
    if downsample > 1:
        quantity = quantity[::downsample, ::downsample]
        roi_x = roi.x[::downsample]
        roi_y = roi.y[::downsample]
    else:
        roi_x = roi.x
        roi_y = roi.y

    my_extent = [min(roi_y), max(roi_y), min(roi_x), max(roi_x)]

    if positive:
        mesh = plot.imshow(quantity, cmap=cmap, vmax=vabsmax, vmin=0, origin='lower', extent=my_extent)
    elif vabsmax == -1:
        mesh = plot.imshow(quantity, cmap=cmap, origin='lower', extent=my_extent)
    else:
        mesh = plot.imshow(quantity, cmap=cmap, vmax=vabsmax, vmin=-vabsmax, origin='lower', extent=my_extent)
    
    if is_plt: 
        plt.xlabel('$x$ [$c/\omega_{p}$]')
        plt.ylabel('$y$ [$c/\omega_{p}$]')
        plt.xlim(my_extent[:2])
        plt.ylim(my_extent[2:])
        plt.colorbar()
    else:
        plot.set_ylabel('$x$ [$c/\omega_{p}$]')
        plot.set_xlabel('$y$ [$c/\omega_{p}$]')
        plot.set_ylim(my_extent[:2])
        plot.set_xlim(my_extent[2:])

    #
    plt.gca().set_aspect('equal')

    if title is not None:
        plt.title(f"{title}")

In [ ]:
print(0 / 0)

# Plot Domain and Spectrum

counter = 0
ds = 10

flh = FieldLineHelper(run, 3, contour_ds = ds) #20, 0.1, 0.1

for abs_ts in my_steps:
    print(abs_ts)
    plt.close()

    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10, 8))

    buffer = 5
    rect = [buffer, run.xlim - buffer, buffer, run.ylim - buffer]

    roi = fq_cut(d.fields, abs_ts - my_steps[0], rect)


    mesh = plot_fq4(roi, roi.dens, plot = ax, vabsmax = 5, downsample = ds, positive = True)
    cbar1 = fig.colorbar(mesh, ax=ax)
    cbar1.set_label('$n / n_0$', rotation=270, labelpad=15)

    #ez_rep = roi.ez.mean()
    #ez_rep = np.mean(roi.ez[0, :])
    ez_rep = roi.ez[0, 0]
    flh.plot_field_lines(roi, ez_rep, arrow_spacing = 7 * 50)
    
    #print(ez_rep.values.item() / (run.B0 * run.beta_A))

    xa, ya = [0.02, 0.99]

    #ax.text(xa, ya, f"$\sigma$ = {run.sigma}", color='black', ha='left', va='top',
    #        transform=ax.transAxes, fontsize = 20, weight = "bold",
    #        bbox=dict(boxstyle='square,pad=0.3', edgecolor='white', facecolor='white', alpha=0.8))
    
    plt.title(f"$\Omega t$ = {round(abs_ts * run.gyrofreq):05d}", fontsize = 20)

    save_movie_frame(counter, dpi = 300)
    counter +=1